# Forest clustering comparison on Titanic

This notebook compares `ForestClusterer`, three tree-based estimators, and two sklearn baselines on a reproducible 300-row sample of Titanic. A repository-local CSV is used when available; otherwise the notebook downloads the public CSV mirror shown in the loading cell.

Algorithms compared:
- `ForestClusterer` - existing random partition embedding implementation.
- `UnsupervisedRandomForestClusterer` - Breiman-style real-vs-synthetic random forest proximity.
- `ExtraTreesProximityClusterer` - ExtraTrees real-vs-synthetic leaf proximity.
- `UnsupervisedBinaryTreeClusterer` - greedy unsupervised CART-like binary partition tree.
- `KMeans` and `AgglomerativeClustering` - sklearn baselines on the same preprocessed features.

All methods are asked for three clusters, an exploratory choice rather than known truth. Internal metrics are computed in one declared OHE + scaled space. Survival is excluded from the features and used only as a weak external reference; it is not a clustering ground truth. The 300-row limit keeps the dense proximity estimators inexpensive and is not a scalability benchmark.


In [ ]:
from pathlib import Path
import sys
import time

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from forest_clustering import (
    ForestClusterer,
    UnsupervisedRandomForestClusterer,
    ExtraTreesProximityClusterer,
    UnsupervisedBinaryTreeClusterer,
)

np.set_printoptions(precision=4, suppress=True)
print('Project root:', PROJECT_ROOT)


In [ ]:
TITANIC_CANDIDATES = [
    PROJECT_ROOT / 'data' / 'titanic.csv',
]
TITANIC_URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'

for path in TITANIC_CANDIDATES:
    if path.exists():
        df = pd.read_csv(path)
        print('Loaded Titanic from:', path)
        break
else:
    df = pd.read_csv(TITANIC_URL)
    print('Loaded Titanic from:', TITANIC_URL)

wanted = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
cols = [c for c in wanted if c in df.columns]
sample = df.sample(n=min(300, len(df)), random_state=123).reset_index(drop=True)
X = sample[cols].copy()
y_survived = sample['Survived'].to_numpy() if 'Survived' in sample.columns else None

print('Shape:', X.shape)
display(X.head())
print('Missing values:')
display(X.isna().sum().to_frame('missing'))


In [ ]:
def make_onehot():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_cols = [c for c in X.columns if c not in numeric_cols]

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', make_onehot())]), categorical_cols),
], remainder='drop')

t0 = time.perf_counter()
X_eval = preprocessor.fit_transform(X)
preprocessing_time = time.perf_counter() - t0
if hasattr(X_eval, 'toarray'):
    X_eval = X_eval.toarray()
print('Evaluation matrix shape:', X_eval.shape)


In [ ]:
def safe_silhouette(X_eval, labels):
    unique = np.unique(labels)
    if len(unique) < 2 or len(unique) >= len(labels):
        return np.nan
    return silhouette_score(X_eval, labels)

def model_summary(name, labels, seconds, fit_obj=None):
    labels = np.asarray(labels)
    row = {
        'algorithm': name,
        'n_clusters_found': int(len(np.unique(labels))),
        'cluster_sizes': dict(zip(*np.unique(labels, return_counts=True))),
        'silhouette': safe_silhouette(X_eval, labels),
        'calinski_harabasz': calinski_harabasz_score(X_eval, labels) if len(np.unique(labels)) > 1 else np.nan,
        'davies_bouldin': davies_bouldin_score(X_eval, labels) if len(np.unique(labels)) > 1 else np.nan,
        'fit_time_s': seconds,
    }
    if y_survived is not None:
        row['ARI_vs_survived'] = adjusted_rand_score(y_survived, labels)
        row['NMI_vs_survived'] = normalized_mutual_info_score(y_survived, labels)
    if fit_obj is not None and hasattr(fit_obj, 'proximity_matrix'):
        P = fit_obj.proximity_matrix()
        row['proximity_shape'] = str(P.shape)
        row['proximity_diag_mean'] = float(np.diag(P).mean())
    return row


In [ ]:
models = []

# Comparable deterministic budgets; all methods are asked for exactly three clusters.
models.append(('ForestClusterer', ForestClusterer(n_iterations=200, n_bins=3, quantile_cuts=True, n_clusters=3, random_state=123, corr_threshold=0.9, n_jobs=1)))
models.append(('Breiman URF', UnsupervisedRandomForestClusterer(n_estimators=200, n_clusters=3, min_samples_leaf=2, random_state=123, n_jobs=1)))
models.append(('ExtraTrees Proximity', ExtraTreesProximityClusterer(n_estimators=200, n_clusters=3, min_samples_leaf=2, random_state=123, n_jobs=1)))
models.append(('Unsupervised Binary Tree', UnsupervisedBinaryTreeClusterer(n_clusters=3, min_samples_leaf=10, min_samples_split=20, max_depth=5, n_thresholds=32, random_state=123)))

rows = []
labels_by_name = {}
for name, model in models:
    t0 = time.perf_counter()
    labels = model.fit_predict(X)
    elapsed = time.perf_counter() - t0
    labels_by_name[name] = labels
    rows.append(model_summary(name, labels, elapsed, model))
    print(f'{name}: labels={np.unique(labels).tolist()}, sizes={dict(zip(*np.unique(labels, return_counts=True)))}')

# sklearn baselines on identical preprocessed features
km = KMeans(n_clusters=3, random_state=123, n_init=10)
t0 = time.perf_counter()
labels = km.fit_predict(X_eval)
elapsed = preprocessing_time + time.perf_counter() - t0
labels_by_name['sklearn KMeans'] = labels
rows.append(model_summary('sklearn KMeans', labels, elapsed))

try:
    agg = AgglomerativeClustering(n_clusters=3, metric='euclidean', linkage='ward')
except TypeError:
    agg = AgglomerativeClustering(n_clusters=3, affinity='euclidean', linkage='ward')
t0 = time.perf_counter()
labels = agg.fit_predict(X_eval)
elapsed = preprocessing_time + time.perf_counter() - t0
labels_by_name['sklearn Agglomerative Ward'] = labels
rows.append(model_summary('sklearn Agglomerative Ward', labels, elapsed))

results = pd.DataFrame(rows)
# Make dict columns readable in both notebook and PDF.
results['cluster_sizes'] = results['cluster_sizes'].map(lambda d: ', '.join(f'{k}: {v}' for k, v in d.items()))
display(results)
best_silhouette = results.loc[results['silhouette'].idxmax()]
best_survival_ari = results.loc[results['ARI_vs_survived'].idxmax()]
print(f"Best common-space silhouette: {best_silhouette['algorithm']} "
      f"({best_silhouette['silhouette']:.3f})")
print(f"Highest weak-reference ARI: {best_survival_ari['algorithm']} "
      f"({best_survival_ari['ARI_vs_survived']:.3f})")


In [ ]:
plot_df = results.set_index('algorithm')[['silhouette', 'ARI_vs_survived' if y_survived is not None else 'calinski_harabasz']]
ax = plot_df.plot(kind='bar', figsize=(10, 4), rot=35)
ax.set_title('Titanic clustering diagnostics')
ax.set_xlabel('Algorithm')
ax.set_ylabel('Score')
plt.tight_layout()
plt.show()


In [ ]:
# Pairwise ARI between algorithm outputs: useful to see whether methods partition data similarly.
names = list(labels_by_name)
ari_matrix = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        ari_matrix.loc[a, b] = adjusted_rand_score(labels_by_name[a], labels_by_name[b])
display(ari_matrix.round(3))


In [ ]:
bt = [m for name, m in models if name == 'Unsupervised Binary Tree'][0]
rules_df = pd.DataFrame(bt.rules())
display(rules_df)


## Interpretation

All internal scores above use the same OHE + scaled feature space. This is transparent and comparable, but it naturally favours algorithms optimized for Euclidean geometry; proximity methods should additionally be evaluated in their native distance space and by repeated-seed stability. The notebook intentionally does not mix native-space silhouettes into the same ranking table.

Read the metrics separately:

- higher silhouette and Calinski–Harabasz, and lower Davies–Bouldin, indicate tighter separation in the declared evaluation space;
- ARI/NMI versus survival measure association with a held-out attribute, not intrinsic cluster quality; a method can score well there and poorly on silhouette;
- pairwise ARI shows whether algorithms agree with each other, not whether any of them is correct; and
- proximity diagonal equal to one is only an API invariant, not evidence of useful off-diagonal structure.

The binary tree is the easiest estimator to explain because the final cell exposes root-to-leaf rules. The forest and ExtraTrees estimators aggregate many leaf co-occurrences, but that richer representation is not automatically a better partition. In the checked execution, sklearn Ward has the highest common-space silhouette (`0.344`), followed by sklearn KMeans (`0.336`) and the binary tree (`0.308`); ForestClusterer reaches `0.144` and Breiman URF is negative. The highest survival ARI is only `0.133` and belongs to sklearn KMeans. The standard baselines therefore win this small comparison; a weak external association must not be used to relabel a geometrically poor partition as good.
